# Reference-data generation (pyscf / gpu4pyscf)

Folder-staged pipeline: each stage cell drains its input folder into the next, so a
molecule advances simply by running cells top-to-bottom. Stages are **idempotent and
resumable** — re-running a cell skips items already produced, so a Colab reconnect just
means re-running the cells.

```
00_inbox  ->  01_optimized  ->  02_frequencies  ->  03_wigner  ->  04_labeled
```

Per molecule: optimize (wB97M-V/def2-svpd) → harmonic frequencies → Wigner sampling
(300 K, 500 structures) → label each with energy/forces/dipole/quadrupole/
polarizability/dipole-derivatives → extended-XYZ matching `data/labels/*.extxyz`.

The compute core prefers **gpu4pyscf** (Colab GPU) and falls back to **pyscf** (local CPU).

## 1. Install
On Colab (GPU) this installs the gpu4pyscf stack; locally it installs the CPU pyscf backend.
Either way it then installs the `rsfff` package in editable mode.

In [1]:
import sys, subprocess, os, shutil, importlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # GPU stack (mirrors notebooks/almo_eda_pyscf.ipynb) + geomeTRIC for opt.
    # Analytic CPSCF response comes from gpu4pyscf.properties on this path.
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "pyscf==2.8.0", "basis-set-exchange==0.11", "geometric==1.1.0",
        "cupy-cuda12x==13.4.1", "cutensor-cu12==2.2.0",
        "gpu4pyscf-cuda12x==1.7.6"], check=True)

    # Do NOT name the clone directory "rsfff". Colab puts /content on sys.path, so
    # a bare ./rsfff directory becomes an implicit *namespace package* that merges
    # with the installed one: submodules still import, but rsfff.__file__ is None.
    # Cloning to a non-colliding name keeps rsfff a normal package.
    if os.path.isdir(os.path.join("rsfff", ".git")):
        shutil.rmtree("rsfff")
        print("removed ./rsfff clone (its name shadowed the rsfff package)")

    REPO_DIR, REPO_URL = "rsfff_repo", "https://github.com/heindelj/rsfff.git"
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                          capture_output=True, text=True).stdout.strip()
    print("rsfff repo at commit:", head)

    # --no-deps is important: rsfff's full dependency list includes torch-cluster,
    # which has no Colab wheel and compiles from source (20-40+ min). rsfff.qcgen
    # needs only numpy + pyscf, both already installed above, so we install the
    # package itself and skip dependency resolution entirely.
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "--no-deps"], check=True)
    REPO_PATH = Path(REPO_DIR).resolve()
else:
    # Local CPU backend. VV10 (needed by wB97M-V) is built into pyscf; we do NOT
    # install pyscf-dispersion (D3/D4 only, and its release is broken vs new numpy).
    # pyscf-properties supplies the analytic CPHF polarizability on CPU; joblib
    # parallelizes the label stage (the macOS pyscf wheel has no OpenMP, so a
    # single SCF is stuck on one core no matter what OMP_NUM_THREADS says).
    subprocess.run([sys.executable, "-m", "pip", "install", "pyscf>=2.14",
                    "geometric", "pyscf-properties", "joblib>=1.3"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".."], check=True)
    REPO_PATH = Path("..").resolve()

# Drop any stale/shadowed cached import of rsfff so the next cell picks up the
# version we just installed (no kernel restart needed).
for _m in [k for k in list(sys.modules) if k == "rsfff" or k.startswith("rsfff.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# REPO_PATH is the single source of truth for repo-relative data (see the seed
# cell); it never depends on rsfff.__file__, which is None under a namespace
# package, nor on the notebook's CWD.
assert (REPO_PATH / "data" / "mols").is_dir(), f"no data/mols under {REPO_PATH}"
print("install cell done; IN_COLAB =", IN_COLAB)
print("repo :", REPO_PATH)

Obtaining file:///Users/joseph.heindel/dev/rsfff
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for rsfff (pyproject.toml): started
  Building editable for rsfff (pyproject.toml): finished with status 'done'
  Created wheel for rsfff: filename=rsfff-0.1.0-0.editable-py3-none-any.whl size=3088 sha256=1cfa6d32cb03aa114c114c6835ff705650523184809c615768e68a9e257cf5a9
  Stored in directory: /private/var/folders/g1/f38hz3r94jl6s36kkjb54z100000gp/T/pip-ephem-wheel-cache-skh86t1e/wheels/dd/3a/38/58a7b35bf1a97a4f03726fffb87b308b45873e

## 2. Setup
Pick the pipeline root (Colab Drive folder or a local directory), create the stage folders,
and report which backend is active.

In [ ]:
import os
# macOS libomp guard (harmless elsewhere); must be set before importing pyscf.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pathlib import Path
from rsfff.qcgen import pipeline as pl
from rsfff.qcgen.backend import backend_name

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/QC_runs")
else:
    # Local: keep runs under the repo's data/ tree.
    ROOT = Path("..") / "data" / "qc_runs"

cfg = pl.PipelineConfig(
    root=ROOT,
    xc="wB97M-V",
    basis="def2-svpd",
    temperature=300.0,
    n_samples=500,
    seed=0,
    # Analytic coupled-perturbed SCF for the polarizability (3 linear solves
    # instead of 6 SCF runs). "finite-difference" for an independent cross-check.
    response="cpscf",
    # Dipole derivatives (atomic polar tensors) are OFF: they dominated labeling
    # cost (a Hessian object + 3N coupled-perturbed solves, vs 3 for everything
    # else). With them off the dipole_derivatives key is absent from the extxyz
    # header. Flip to True if you want them back.
    with_dipole_derivatives=False,
    # Worker processes for the label stage (structures are independent). Held at
    # 10 rather than every core, to leave headroom for other work on the machine.
    # Ignored on the GPU backend, which is pinned to 1 (a single device).
    n_workers=10,
)
pl.make_dirs(cfg)
print("backend :", backend_name())
print("root    :", cfg.root.resolve())
print("method  :", cfg.xc, "/", cfg.basis, "| T =", cfg.temperature, "K | N =", cfg.n_samples)
print("response:", cfg.response, "| dipole derivatives:", cfg.with_dipole_derivatives)
print("workers :", pl.resolve_workers(cfg))

## 3. Seed the inbox
Copy the molecules you want to process into `00_inbox/`. Charges/multiplicities are looked up
in `pipeline.SPECIES` (open-shell allowed, e.g. `oh_radical` → doublet). Edit `NAMES` to choose
which species to run; `None` seeds everything in `data/mols/`.

In [3]:
# REPO_PATH comes from the install cell and works on Colab and locally alike.
# (Deliberately not derived from rsfff.__file__, which is None when a same-named
# directory on sys.path turns rsfff into a namespace package.)
MOLS = REPO_PATH / "data" / "mols"

NAMES = ["h2o"]  # subset to run; None = every *.xyz in data/mols

if NAMES is None:
    xyz_paths = sorted(MOLS.glob("*.xyz"))
else:
    xyz_paths = [MOLS / f"{n}.xyz" for n in NAMES]

missing = [p.name for p in xyz_paths if not p.exists()]
assert not missing, f"missing geometries in {MOLS}: {missing}"

seeded = pl.seed_inbox(cfg, xyz_paths)
print(f"seeded {len(seeded)} file(s) into {cfg.root / pl.INBOX}:")
print(" ", ", ".join(seeded) if seeded else "(nothing new; inbox already populated)")
# You can also drop your own .xyz files directly into the inbox folder above and
# the stages will pick them up. On Colab that folder lives on Google Drive
# (ROOT = /content/drive/MyDrive/QC_runs), so inputs and outputs both persist
# across sessions -- no separate Drive plumbing needed.

seeded 0 file(s) into ../data/qc_runs/00_inbox:
  (nothing new; inbox already populated)


## 4. Stage: optimize geometries
`00_inbox/*.xyz` → `01_optimized/<name>.xyz` (+ `<name>.json` meta).

In [4]:
pl.run_stage(pl.STAGE_OPTIMIZE, cfg)

[16:31:22] stage 'optimize' [pyscf]: 1 candidate(s) in 00_inbox/
[16:31:22] stage 'optimize': 0 done, 1 skipped, 0 failed


([], ['h2o'], [])

## 5. Stage: harmonic frequencies
`01_optimized/` → `02_frequencies/<name>.npz` (Hessian, masses, geometry). Analytic Hessian
where supported; automatic finite-difference fallback for UKS + NLC (open-shell wB97M-V).

In [5]:
pl.run_stage(pl.STAGE_FREQUENCIES, cfg)

[16:31:22] stage 'frequencies' [pyscf]: 1 candidate(s) in 01_optimized/
[16:31:22] stage 'frequencies': 0 done, 1 skipped, 0 failed


([], ['h2o'], [])

## 6. Stage: Wigner sampling
`02_frequencies/` → `03_wigner/<name>.extxyz` (`n_samples` geometries at `temperature`).

In [6]:
pl.run_stage(pl.STAGE_WIGNER, cfg)

[16:31:22] stage 'wigner' [pyscf]: 1 candidate(s) in 02_frequencies/
[16:31:22] stage 'wigner': 0 done, 1 skipped, 0 failed


([], ['h2o'], [])

## 7. Stage: label properties
`03_wigner/` → `04_labeled/<name>.extxyz` (energy, forces, dipole, quadrupole,
polarizability, dipole derivatives). On success the original inbox `.xyz` is archived to
`_completed/`. This is the expensive stage; it streams frames to disk and writes atomically.

In [7]:
pl.run_stage(pl.STAGE_LABEL, cfg)

[16:31:22] stage 'label' [pyscf]: 1 candidate(s) in 03_wigner/
[16:31:22]   label: h2o ...
[16:31:22]     h2o: labeling 500 frames on 13 worker(s)


/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/rhf.py:45: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/rhf.py:45: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/rhf.py:45: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/rhf.py:45: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/dhf.py:30: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG is under testing')
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/pyscf/prop/efg/dhf.py:30: UserWarning: Module EFG is under testing
  warnings.warn('Module EFG

KeyboardInterrupt: 

## 8. Collect + summary
Concatenate all labeled species into one dataset file and report per-species frame counts and
any failures (tracebacks live in `_failed/`).

In [ ]:
out_path, counts = pl.collect(cfg)
print("combined dataset ->", out_path.resolve())
for name, n in sorted(counts.items()):
    print(f"  {name:14s} {n} frames")

failed = sorted((cfg.root / pl.FAILED).glob("*.error.log"))
if failed:
    print("\nfailures:")
    for f in failed:
        print("  ", f.name)